# W05-装饰器与上下文

什么是闭包？

就是把 函数 和 函数外部的变量 一起带走


In [2]:
def maker_add(n):
    def add(x):
        return x + n
    return add

print(maker_add(5)(3))


8


什么是装饰器？

在原有的函数基础上添加一些功能


In [6]:
def world(fn):
    def add_world():
        return fn() + " world!"
    return add_world

# method 1
def hello():
    return "hello"

hello = world(hello)
print(hello())

# method 2
@world
def hi():
    return "hi"

print(hi())


hello world!
hi world!


## 异常处理体系

| 子句 | 何时执行 |
| --------- | ---------- |
| `try` | 主代码 |
| `except` | 出错时 |
| `else` | 没出错时 |
| `finally` | 永远执行（清理资源） |

In [7]:
try:
    n = int(input("数字："))
    print(10 / n)
except ValueError:
    print("不是有效数字")
except ZeroDivisionError:
    print("除以零了")
except Exception as e:           # 兜底（不要省略变量 e）
    print("其他错误", e)
else:
    print("没异常才执行")
finally:
    print("无论如何都执行（清理）")
    

数字： 


不是有效数字
无论如何都执行（清理）


## 实战 1：手写 5 个装饰器

In [14]:
import time, json
from functools import wraps

# 1.tiemr
def timer(fn):
    @wraps(fn)
    def wrapper(*a, **kw):
        s = time.perf_counter()
        r = fn(*a, **kw)
        print(f"[timer] {fn.__name__} {time.perf_counter()-s:.4f}s")
        return r
    return wrapper

# 2.retry
def retry(tiems=3):
    def deco(fn):
        @wraps(fn)
        def wrapper(*a, **kw):
            for i in range(times):
                try:
                    return fn(*a, **kw)
                except Exception as e:
                    print(f"第{i+1}次失败 {e}")
            raise
        return wrapper
    return deco

# 3. cache
def cache(fn):
    store = {}
    @wraps(fn)
    def wrapper(*a):
        key = json.dumps(a, default=str)
        if key not in store:
            store[key] = fn(*a)
        return store[key]
    return wrapper

# 4.deprecated
def deprecated(fn):
    def deco(fn):
        @wraps(fn)
        def wrapper(*a, **kw):
            print(f"[弃用] {fn.__name__} {reason}")
            return fn(*a, **kw)
        return wrapper
    return deco

# 5.validate
def validate(**rules):
    def deco(fn):
        @wraps(fn)
        def wrapper(*a, **kw):
            for k, t in rules.items():
                if k in kw and not isinstance(kw[k], t):
                    raise TypeError(f"{k} 必须是 {t.__name__}")
            return fn(*a, **kw)
        return wrapper
    return deco

@timer
@cache
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

@validate(n=int)
def square(*, n):
    return n*n

print(fib(20))
print(square(n=5))


[timer] fib 0.0000s
[timer] fib 0.0000s
[timer] fib 0.0002s
[timer] fib 0.0000s
[timer] fib 0.0002s
[timer] fib 0.0000s
[timer] fib 0.0002s
[timer] fib 0.0000s
[timer] fib 0.0009s
[timer] fib 0.0000s
[timer] fib 0.0010s
[timer] fib 0.0000s
[timer] fib 0.0010s
[timer] fib 0.0000s
[timer] fib 0.0010s
[timer] fib 0.0000s
[timer] fib 0.0010s
[timer] fib 0.0000s
[timer] fib 0.0010s
[timer] fib 0.0000s
[timer] fib 0.0010s
[timer] fib 0.0000s
[timer] fib 0.0011s
[timer] fib 0.0000s
[timer] fib 0.0011s
[timer] fib 0.0000s
[timer] fib 0.0011s
[timer] fib 0.0000s
[timer] fib 0.0011s
[timer] fib 0.0000s
[timer] fib 0.0011s
[timer] fib 0.0000s
[timer] fib 0.0011s
[timer] fib 0.0000s
[timer] fib 0.0012s
[timer] fib 0.0000s
[timer] fib 0.0012s
[timer] fib 0.0000s
[timer] fib 0.0012s
6765
25


## 实战 2：文件操作上下文管理器

In [18]:
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def safe_writer(path, encoding='utf-8'):
    """写入临时文件，成功才 rename — 防止半截文件覆盖原文件。"""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    f = tmp.open("w", encoding=encoding)
    try:
        yield f
        f.close()
        tmp.replace(path)
    except Exception as e:
        f.close()
        tmp.unink(missing_ok=True)
        print(f"Error: {e}")
        raise

with safe_writer("helloworld.txt") as f:
    f.write('{"debug": true}')
print("finish")


finish


## 实战 3：安全的资源释放

In [20]:
from contextlib import contextmanager

class FakeDB:
    def __init__(self):
        self.connected = False

    def connect(self):
        self.connected = True
        print("The DB is connected")

    def close(self):
        self.connected = False
        print("The DB is closed")

    def query(self, sql):
        print("DO: ", sql)


@contextmanager
def db_session():
    db = FakeDB()
    db.connect()
    try:
        yield db
    finally:
        db.close()

with db_session() as db:
    db.query("SELECT * FROM users")


The DB is connected
DO:  SELECT * FROM users
The DB is closed
